# Módulo 05 — Memória Associativa (Regra de Hebb)
**Portfólio de Laboratório de RNA**
**Aluno:** Gabriel Rocha Guimarães | **RA:** 23110134 | **Turma:** 26-1-COMP-7-07-B

---

## Fundamentação Teórica

Postulada por Donald Hebb em 1949, a **Regra de Hebb** captura um princípio biológico fundamental: *"neurônios que disparam juntos, fortalecem suas conexões."* Matematicamente, a matriz de pesos é construída como soma de produtos externos:

$$W = \sum_{k=1}^{p} \mathbf{y}_k \mathbf{x}_k^T$$

Cada termo $\mathbf{y}_k \mathbf{x}_k^T$ codifica a associação entre o padrão de entrada $\mathbf{x}_k$ e o padrão de saída $\mathbf{y}_k$.

**Memória heteroassociativa:** associa padrões de dois domínios distintos — dado $\mathbf{x}$, recupera $\mathbf{y}$. A recuperação é direta:

$$\mathbf{y}_{rec} = W \cdot \mathbf{x}$$

**Memória autoassociativa:** entrada e saída pertencem ao mesmo espaço ($\mathbf{x} = \mathbf{y}$), permitindo recuperar um padrão completo a partir de uma versão incompleta ou ruidosa.

O aprendizado de Hebb é **não-iterativo e one-shot** — uma única apresentação de cada par é suficiente para codificar a associação na matriz.

In [1]:
# Passo 3 — Regra de Hebb: memória heteroassociativa
import numpy as np

# Pares de associação (x → y) — vetores coluna
x1 = np.array([[1],[0]]); y1 = np.array([[1],[0],[0]])
x2 = np.array([[0],[1]]); y2 = np.array([[0],[1],[0]])

# Matrizes de Hebb (produto externo)
w1 = np.dot(y1, x1.T)
w2 = np.dot(y2, x2.T)
W = w1 + w2

print('Matriz de pesos W (construída por Hebb):')
print(W)
print(f'Dimensão: {W.shape} (saída x entrada)')

# Teste de recuperação exata
print('\nTeste de recuperação:')
for x, y_esperado in [(x1, y1), (x2, y2)]:
    recuperado = np.dot(W, x)
    print(f'  Entrada: {x.flatten()} → Recuperado: {recuperado.flatten()} | Esperado: {y_esperado.flatten()}')

# Teste com entrada ligeiramente ruidosa
teste_ruidoso = np.array([[0.9],[0.1]])
recuperado_ruidoso = np.dot(W, teste_ruidoso)
print(f'\nEntrada ruidosa [0.9, 0.1]: recuperado = {recuperado_ruidoso.flatten().round(3)}')
print(f'Valor máximo aponta para saída {recuperado_ruidoso.argmax()} (correto: 0)')

Matriz de pesos W (construída por Hebb):
[[1 0]
 [0 1]
 [0 0]]
Dimensão: (3, 2) (saída x entrada)

Teste de recuperação:
  Entrada: [1 0] → Recuperado: [1 0 0] | Esperado: [1 0 0]
  Entrada: [0 1] → Recuperado: [0 1 0] | Esperado: [0 1 0]

Entrada ruidosa [0.9, 0.1]: recuperado = [0.9 0.1 0. ]
Valor máximo aponta para saída 0 (correto: 0)


In [2]:
# Passo 4 — Exercício: Formas → Nomes (Shapes H/V)
import numpy as np

# Formas geométricas como vetores de entrada (2D)
H = np.array([[1], [0]])  # forma Horizontal
V = np.array([[0], [1]])  # forma Vertical

# Nomes como vetores de saída (4D) — codificação one-hot
nome_Ana    = np.array([[1], [0], [0], [0]])  # Ana = horizontal
nome_Bruno  = np.array([[0], [1], [0], [0]])  # Bruno = vertical

# Construir memória por regra de Hebb
wH = np.dot(nome_Ana, H.T)
wV = np.dot(nome_Bruno, V.T)
W_formas = wH + wV

print('Matriz W (formas → nomes):')
print(W_formas)

print('\nTestes de recuperação:')
for forma, nome_esp, label in [(H, nome_Ana, 'H → Ana'), (V, nome_Bruno, 'V → Bruno')]:
    resultado = np.dot(W_formas, forma)
    match = np.array_equal(resultado, nome_esp)
    print(f'  {label}: {resultado.flatten()} | Correto: {match}')

# Teste com ruído
H_ruidoso = np.array([[0.8], [0.2]])
res_ruidoso = np.dot(W_formas, H_ruidoso)
print(f'\nH ruidoso [0.8, 0.2]: {res_ruidoso.flatten()}')
print(f'  Índice máximo: {res_ruidoso.argmax()} → Ana (correto)')

Matriz W (formas → nomes):
[[1 0]
 [0 1]
 [0 0]
 [0 0]]

Testes de recuperação:
  H → Ana: [1 0 0 0] | Correto: True
  V → Bruno: [0 1 0 0] | Correto: True

H ruidoso [0.8, 0.2]: [0.8 0.2 0.  0. ]
  Índice máximo: 0 → Ana (correto)


## Análise Crítica

**Construção sem iteração:** Diferente de todos os outros modelos do portfólio, a matriz de Hebb é construída analiticamente em uma única passagem. Não há gradiente, não há épocas — cada par é processado uma vez e codificado permanentemente.

**Ortogonalidade e recuperação perfeita:** Quando os vetores de entrada são ortogonais (como `[1,0]` e `[0,1]`), a recuperação é matematicamente exata: $W \cdot x_1 = y_1$ sem interferência do outro padrão. A ortogonalidade garante que as memórias não se contaminem.

**Interferência entre padrões:** Quando os padrões de entrada não são ortogonais, a recuperação sofre interferência cruzada — o produto externo de um par "vaza" para a recuperação do outro. Esse é o principal limite da memória de Hebb linear.

**Tolerância a ruído:** A memória heteroassociativa tolera pequenas perturbações nas entradas — um vetor ruidoso ainda produz uma saída onde o índice de maior magnitude aponta para a resposta correta. Para uma robustez maior, as Redes de Hopfield (Módulo 06) introduzem recuperação iterativa.